# Tabulate phenotypes of SARS-CoV-2 spike that might have predictive power in understanding evolution for different mutations and lineages

Import Python modules:

In [30]:
import datetime
import gzip
import json
import os
import urllib.request
import numpy
import altair as alt
import pandas as pd
import ruamel.yaml as yaml
_ = alt.data_transformers.disable_max_rows()

Read the configuration YAML:

In [31]:
# This cell is tagged as parameters. So if you run the notebook using papermill
# (https://papermill.readthedocs.io/en/latest/usage-cli.html) with
# `-p config_yaml <config_yaml_file>` then the file defined here will be replaced
# with the one you pass
config_yaml = "XBB_config.yaml"

In [32]:
with open(config_yaml) as f:
    config = yaml.YAML().load(f)

Split Spike Pseudovirus DMS data into Spike RBD (RBD) and Spike Non-RBD (S2) phenotypes:

In [34]:
# Read in raw spike_pseudovirus_DMS_XBB.1.5.csv phenotype data
spike_raw = pd.read_csv(config["mutation_phenotype_csvs"]["XBB.1.5"]["spike pseudovirus DMS"]["csv_raw"])
phenotypes_raw = list(config["mutation_phenotype_csvs"]["XBB.1.5"]["spike pseudovirus DMS"]["columns_raw"])

# Split by region: (RBD, S2, NTD, other) and add new column names
spike_pivot = spike_raw.pivot(index=["site", "wildtype", "mutant", "sequential_site"], columns="region", values=phenotypes_raw)
spike_pivot.columns = [f'{region} {col}' for col, region in spike_pivot.columns]
spike_pivot = spike_pivot.reset_index()
spike_pivot.to_csv(config["spike_pseudovirus_dms_pivot"], sep=",")

## Mutation effects on phenotypes
Read all the specified mutation-level phenotypic effects into a Data Frame.
Write that data frame to a file.
Also created data frame and file in which the mutation-level phenotypic data are randomized among mutations.

In [35]:
mutation_phenotypes = []
req_cols = ["site", "wildtype", "mutant"]
ref_lineages = []
for ref_lineage, ref_lineage_d in config["mutation_phenotype_csvs"].items():
    ref_lineages.append(ref_lineage)
    for phenotype_category, category_d in ref_lineage_d.items():
        pheno_cols = category_d["columns"]
        print(
            f"Reading {phenotype_category} phenotypes from {category_d['csv']}\n  "
            + "\n  ".join(pheno_cols)
        )
        if set(req_cols).intersection(pheno_cols):
            raise ValueError(f"{phenotype_category} columns {pheno_cols} include {req_cols}")
        df = pd.read_csv(category_d["csv"])
        if not set(req_cols + pheno_cols).issubset(df.columns):
            raise ValueError(f"cannot find expected columns for {phenotype_category}")
        mutation_phenotypes.append(
            df
            .melt(
                id_vars=req_cols,
                value_vars=pheno_cols,
                var_name="phenotype",
                value_name="mutation_effect",
            )
            .assign(
                ref_lineage=ref_lineage,
                phenotype=lambda x: (
                    phenotype_category
                    if len(pheno_cols) == 1 and pheno_cols[0] == phenotype_category
                    else phenotype_category + " " + x["phenotype"]
                ),
            )
            [["ref_lineage", "phenotype", *req_cols, "mutation_effect"]]
            .query("mutation_effect.notnull()")
            .query("wildtype != mutant")  # drop any wildtype to wildtype mutations
        )

mutation_phenotypes = pd.concat(mutation_phenotypes, ignore_index=True)

mutation_phenotypes_csv = config["mutation_phenotypes_csv"]
os.makedirs(os.path.dirname(mutation_phenotypes_csv), exist_ok=True)
print(f"\nWriting mutation phenotypes to {mutation_phenotypes_csv}")
mutation_phenotypes.to_csv(mutation_phenotypes_csv, index=False, float_format="%.4g")

n_randomizations = config["n_randomizations"]
print(f"\nNow randomizing each mutation phenotype {n_randomizations} times")
mutation_phenotypes_randomized = []
for _, df in mutation_phenotypes.groupby(["ref_lineage", "phenotype"]):
    for random_seed in range(n_randomizations):
        numpy.random.seed(random_seed)
        mutation_phenotypes_randomized.append(
            df
            .assign(
                mutation_effect=lambda x: numpy.random.permutation(x["mutation_effect"]),
                random_seed=random_seed
            )
            [["random_seed"] + df.columns.tolist()]
        )
mutation_phenotypes_randomized = pd.concat(mutation_phenotypes_randomized, ignore_index=True)
mutation_phenotypes_randomized_csv = config["mutation_phenotypes_randomized_csv"]
os.makedirs(os.path.dirname(mutation_phenotypes_randomized_csv), exist_ok=True)
print(f"Writing randomized mutation phenotypes to {mutation_phenotypes_randomized_csv}")
mutation_phenotypes_randomized.to_csv(mutation_phenotypes_randomized_csv, index=False, float_format="%.4g")

Reading spike pseudovirus DMS phenotypes from data/pivot_spike_pseudovirus_DMS_XBB.1.5.csv
  RBD human sera escape
  RBD ACE2 binding
  RBD spike mediated entry
  S2 human sera escape
  S2 ACE2 binding
  S2 spike mediated entry
Reading RBD yeast-display DMS phenotypes from data/yeast_RBD_DMS_XBB.1.5.csv
  ACE2 affinity
  RBD expression
  escape
Reading EVEscape phenotypes from data/EVEscape_XBB_single_mutation_predictions.csv
  EVEscape

Writing mutation phenotypes to results/mutation_phenotypes.csv

Now randomizing each mutation phenotype 100 times
Writing randomized mutation phenotypes to results/mutation_phenotypes_randomized.csv


## Read the Pango lineages
Read Pango lineage definitions come from Cornelius Roemer's GitHub repo ([https://github.com/corneliusroemer/pango-sequences](https://github.com/corneliusroemer/pango-sequences)):

In [25]:
# {'XDV.1'} missing
pango_json = config["pango_json"]
print(f"Reading Pango lineage definitions from {pango_json}")
with urllib.request.urlopen(pango_json) as url:
    pango_lineages = json.load(url)
print(f"Read definitions for {len(pango_lineages)} lineages")

Reading Pango lineage definitions from https://raw.githubusercontent.com/corneliusroemer/pango-sequences/main/data/pango-consensus-sequences_summary.json
Read definitions for 4253 lineages


Read Pango lineage growth rates from Bedford lab repo ([https://github.com/nextstrain/forecasts-ncov/](https://github.com/nextstrain/forecasts-ncov/)):

In [26]:
pango_growth_json = config["pango_growth_json"]
print(f"Reading Pango lineage growth rates from {pango_growth_json}")
with urllib.request.urlopen(pango_growth_json) as url:
    pango_growth_dict = json.loads(gzip.decompress(url.read()))

# convert to data frame
pango_growth = (
    pd.DataFrame(pango_growth_dict["data"])
    .query("location == 'hierarchical'")
    .query("site == 'ga'")
    .pivot_table(index="variant", values="value", columns="ps")
    .reset_index(names="lineage")
    .assign(
        lineage_growth_HDI_95=lambda x: x.apply(
            lambda row: f"{row['HDI_95_lower']:.3g} to {row['HDI_95_upper']:.3g}",
            axis=1,
        ),
    )
    .rename(columns={"median": "lineage growth", "lineage_growth_HDI_95": "lineage growth HDI 95"})
    [["lineage", "lineage growth", "lineage growth HDI 95"]]
    .query("lineage != 'other'")
)
# Comment out to avoid ValueError: some clades have growth data but are not defined {'XDV.1'}
#assert len(pango_growth) == pango_growth["lineage"].nunique()
#extra_growth_lineages = set(pango_growth["lineage"]) - set(pango_lineages)
#if extra_growth_lineages:
#    raise ValueError(f"some lineages have growth data but are not defined {extra_growth_lineages}")

#print(f"Read growth data for {len(pango_growth)} lineages")

Reading Pango lineage growth rates from https://data.nextstrain.org/files/workflows/forecasts-ncov/gisaid/pango_lineages/global/mlr/latest_results.json


Get a data frame of all lineages along with other relevant information:

In [36]:
def is_descendant_of(lineage, ancestor):
    """Returns True iff `lineage` is a descendant of `ancestor`."""
    if pango_lineages[lineage]["parent"] == ancestor:
        return True
    elif pango_lineages[lineage]["parent"]:
        return is_descendant_of(pango_lineages[lineage]["parent"], ancestor)
    else:
        return False

def relative_mutations(lineage_muts, reference_muts):
    """Get mutation in `lineage_muts` relative `reference_muts`."""
    shared_muts = set(lineage_muts).intersection(reference_muts)
    lineage_sites = {
        r: (wt, m) for (wt, r, m) in [tup for tup in lineage_muts if tup not in shared_muts]
    }
    reference_sites = {
        r: (wt, m) for (wt, r, m) in [tup for tup in reference_muts if tup not in shared_muts]
    }
    muts = []
    for r, (wt, m) in lineage_sites.items():
        if r in reference_sites:
            assert wt == reference_sites[r][0]
            muts.append((r, reference_sites[r][1], m))
        else:
            muts.append((r, wt, m))
    for r, (wt, m) in reference_sites.items():
        if r in lineage_sites:
            assert wt == lineage_sites[r][0]
            pass  # already counted
        else:
            muts.append((r, m, wt))
    return [(wt, r, m) for (r, wt, m) in sorted(muts)]

def parse_spike_muts(lineage_d):
    """Parse spike mutations from dict for a lineage."""
    return [
        (mut.split(":")[1][0], int(mut.split(":")[1][1: -1]), mut.split(":")[1][-1])
        for mut in lineage_d["aaSubstitutions"] + lineage_d["aaDeletions"]
        if mut and mut.startswith("S:")
    ]

lineages_df = {}
ref_lineage_spike_muts = {
    ref_lineage: parse_spike_muts(pango_lineages[ref_lineage])
    for ref_lineage in ref_lineages
}

for lineage, lineage_d in pango_lineages.items():
    spike_muts = parse_spike_muts(lineage_d)
    lineages_df[lineage] = {
        "date": lineage_d["designationDate"] if lineage_d["designationDate"] else pd.NA,
        "parent": lineage_d["parent"] if lineage_d["parent"] else pd.NA,
        "spike muts from Wuhan-Hu-1": " ".join(f"{wt}{r}{m}" for (wt, r, m) in spike_muts),
        "number spike muts from Wuhan-Hu-1": len(spike_muts),
        **{
            f"spike muts from {ref_lineage}":
                relative_mutations(spike_muts, ref_lineage_spike_muts[ref_lineage])
            for ref_lineage in ref_lineages
        },
        **{
            f"descendant of {ancestor}": is_descendant_of(lineage, ancestor)
            for ancestor in config["classify_descendants_of"]
        },

    }

lineages_df = pd.DataFrame.from_dict(lineages_df, orient="index").reset_index(names="lineage")

## Assign phenotypes to lineages
Phenotypes are just sum of mutation effects of all mutations with respect to the `spike_muts_relative_to` lineage.
Mutations without measured phenotype effects are ignored (given values of zero).

In [37]:
class PhenotypeAssigner:
    """Assign phenotypes to sets of mutations.

    Parameters
    ----------
    mutation_phenotypes_df : pandas.DataFrame
        Should have columns `site`, `wildtype`, `mutant`, `mutation_effect`.

    """
    def __init__(self, mutation_phenotypes_df):
        assert len(mutation_phenotypes_df) == len(
            mutation_phenotypes_df[["site", "mutant"]].drop_duplicates()
        )
        self.sites = sorted(set(mutation_phenotypes_df["site"]))
        assert len(self.sites) == len(
            mutation_phenotypes_df[["site", "wildtype"]].drop_duplicates()
        )
        self.wts = mutation_phenotypes_df.set_index("site")["wildtype"].to_dict()
        self.effects = {
            site: site_df.set_index("mutant")["mutation_effect"].to_dict()
            for site, site_df in mutation_phenotypes_df.groupby("site")
        }
        for site, wt in self.wts.items():
            assert wt not in self.effects[site]
            self.effects[site][wt] = 0.0

    def phenotype(self, muts):
        """Returns phenotype for list of `muts` as `(wildtype, site, mutant)`."""
        pheno = 0.0
        for wt, site, m in muts:
            if (site in self.effects) and (wt in self.effects[site]) and (m in self.effects[site]):
                pheno += self.effects[site][m] - self.effects[site][wt]
        return pheno


# assign lineage phenotypes, Hamming distance from reference, and mutation lists as strings
lineage_phenos = lineages_df.copy()
new_lineage_pheno_cols = []
for (ref_lineage, phenotype), df in mutation_phenotypes.groupby(
    ["ref_lineage", "phenotype"], # if we do it for many reference lineages, but we dont need that right now
    sort=False,
):
    phenos = PhenotypeAssigner(df)
    new_lineage_pheno_cols.append(
        lineage_phenos[f"spike muts from {ref_lineage}"].map(phenos.phenotype).rename(
            f"{phenotype} relative to {ref_lineage}"
        )
    )
for ref_lineage in ref_lineages:
    col = f"spike muts from {ref_lineage}"
    new_lineage_pheno_cols.append(
        lineage_phenos[col].map(len).rename(f"Hamming distance relative to {ref_lineage}")
    )
    lineage_phenos[col] = lineage_phenos[col].map(
        lambda mutlist: " ".join(f"{wt}{r}{m}" for (wt, r, m) in mutlist)
    )
lineage_phenos = pd.concat([lineage_phenos] + new_lineage_pheno_cols, axis=1)
lineage_phenotypes_csv = config["lineage_phenotypes_csv"]
os.makedirs(os.path.dirname(lineage_phenotypes_csv), exist_ok=True)
print(f"Writing lineage phenotypes to {lineage_phenotypes_csv}")
lineage_phenos.to_csv(lineage_phenotypes_csv, index=False, float_format="%.4g")

# assign lineage phenotypes from randomized data
print(f"\nComputing lineage phenotypes from {n_randomizations} randomizations of mutation phenotypes")
lineage_phenos_randomized = lineages_df.copy()
new_lineage_pheno_randomized_cols = []
for (random_seed, ref_lineage, phenotype), df in mutation_phenotypes_randomized.groupby(
    ["random_seed", "ref_lineage", "phenotype"]
):
    phenos = PhenotypeAssigner(df)
    new_lineage_pheno_randomized_cols.append(
        lineage_phenos_randomized[f"spike muts from {ref_lineage}"].map(phenos.phenotype).rename(
            f"random_{random_seed} {phenotype} relative to {ref_lineage}"
        )
    )
for ref_lineage in ref_lineages:
    col = f"spike muts from {ref_lineage}"
    for random_seed in mutation_phenotypes_randomized["random_seed"].unique():
        new_lineage_pheno_randomized_cols.append(
            lineage_phenos_randomized[col].map(len).rename(
                f"random_{random_seed} Hamming distance relative to {ref_lineage}"
            )
        )
    lineage_phenos_randomized[col] = lineage_phenos_randomized[col].map(
        lambda mutlist: " ".join(f"{wt}{r}{m}" for (wt, r, m) in mutlist)
    )
lineage_phenos_randomized = pd.concat([lineage_phenos_randomized] + new_lineage_pheno_randomized_cols, axis=1)
lineage_phenotypes_randomized_csv = config["lineage_phenotypes_randomized_csv"]
os.makedirs(os.path.dirname(lineage_phenotypes_randomized_csv), exist_ok=True)
print(f"Writing randomized lineage phenotypes to {lineage_phenotypes_randomized_csv}")
lineage_phenos_randomized.to_csv(lineage_phenotypes_randomized_csv, index=False, float_format="%.4g")

Writing lineage phenotypes to results/lineage_phenotypes.csv

Computing lineage phenotypes from 100 randomizations of mutation phenotypes
Writing randomized lineage phenotypes to results/lineage_phenotypes_randomized.csv
